In [2]:
import pandas as pd
import numpy as np
import torch

from scipy.sparse import load_npz

print("Libraries loaded.")

Libraries loaded.


In [3]:
train = pd.read_parquet(
    "../data/processed/train.parquet"
)

validation = pd.read_parquet(
    "../data/processed/validation.parquet"
)

print("Train:", train.shape)
print("Validation:", validation.shape)

Train: (1928949, 5)
Validation: (413345, 5)


# Recreate the exact common evaluation cohort

In [4]:
fair_users = np.load(
    "../data/processed/bpr_eval_users.npy"
)

print(
    "Fair evaluation users:",
    len(fair_users)
)

Fair evaluation users: 2047


# Build validation ground truth

In [5]:
fair_validation = validation[
    validation["user_id"].isin(fair_users)
].copy()

fair_ground_truth = (
    fair_validation
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

print(
    "Validation interactions:",
    len(fair_validation)
)

print(
    "Users:",
    len(fair_ground_truth)
)

Validation interactions: 25549
Users: 2047


# Create validation ground truth

In [6]:
fair_validation = validation[
    validation["user_id"].isin(fair_users)
].copy()

fair_ground_truth = (
    fair_validation
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

print(
    "Fair validation interactions:",
    len(fair_validation)
)

print(
    "Fair evaluation users:",
    len(fair_ground_truth)
)

Fair validation interactions: 25549
Fair evaluation users: 2047


In [7]:
import os

paths = [
    "../models/product_tfidf.npz",
    "../models/product_ids.npy",
    "../models/user_profiles.npz",
    "../models/warm_user_ids.npy",
    "../models/bpr_model.pt"
]

for path in paths:
    print(
        path,
        "→",
        os.path.exists(path)
    )

../models/product_tfidf.npz → True
../models/product_ids.npy → True
../models/user_profiles.npz → True
../models/warm_user_ids.npy → True
../models/bpr_model.pt → True


In [8]:
import os

paths = [
    "../models/product_tfidf.npz",
    "../models/product_ids.npy",
    "../models/user_profiles.npz",
    "../models/warm_user_ids.npy",
    "../models/bpr_model.pt"
]

for path in paths:
    print(
        path,
        "→",
        os.path.exists(path)
    )

../models/product_tfidf.npz → True
../models/product_ids.npy → True
../models/user_profiles.npz → True
../models/warm_user_ids.npy → True
../models/bpr_model.pt → True


# Load the model artifacts

In [9]:
# Load Content-Based artifacts

product_matrix = load_npz(
    "../models/product_tfidf.npz"
)

product_ids = np.load(
    "../models/product_ids.npy"
)

user_profiles = load_npz(
    "../models/user_profiles.npz"
)

warm_user_ids = np.load(
    "../models/warm_user_ids.npy"
)

print("Product matrix:", product_matrix.shape)
print("Product IDs:", len(product_ids))
print("User profiles:", user_profiles.shape)
print("Warm users:", len(warm_user_ids))

Product matrix: (160670, 86490)
Product IDs: 160670
User profiles: (18965, 86490)
Warm users: 18965


# Create mappings

In [10]:
product_id_to_index = {
    item_id: idx
    for idx, item_id in enumerate(product_ids)
}

warm_user_to_index = {
    user_id: idx
    for idx, user_id in enumerate(warm_user_ids)
}

# Load BPR

In [11]:
bpr_data = torch.load(
    "../models/bpr_model.pt",
    map_location="cpu",
    weights_only=False
)

bpr_user_embeddings = (
    bpr_data["user_embedding"]
)

bpr_item_embeddings = (
    bpr_data["item_embedding"]
)

cf_user_to_idx = (
    bpr_data["cf_user_to_idx"]
)

cf_item_to_idx = (
    bpr_data["cf_item_to_idx"]
)

print(
    "BPR user embeddings:",
    bpr_user_embeddings.shape
)

print(
    "BPR item embeddings:",
    bpr_item_embeddings.shape
)

print(
    "BPR users:",
    len(cf_user_to_idx)
)

print(
    "BPR items:",
    len(cf_item_to_idx)
)

BPR user embeddings: torch.Size([27771, 32])
BPR item embeddings: torch.Size([42672, 32])
BPR users: 27771
BPR items: 42672


In [12]:
cf_idx_to_item = {
    idx: item_id
    for item_id, idx in cf_item_to_idx.items()
}

cf_item_ids = np.array([
    cf_idx_to_item[i]
    for i in range(len(cf_idx_to_item))
])

print("BPR item IDs:", len(cf_item_ids))

BPR item IDs: 42672


# Popularity scores

In [13]:
popularity_scores = (
    train
    .groupby("item_id")
    ["interaction_strength"]
    .sum()
    .sort_values(
        ascending=False
    )
)

print(
    popularity_scores.head(10)
)

item_id
461686    2142
5411      1903
309778    1754
370653    1483
257040    1477
369447    1475
7943      1443
298009    1347
48030     1268
335975    1233
Name: interaction_strength, dtype: int64


In [14]:
popularity_dict = popularity_scores.to_dict()

# Training history

In [15]:
seen_items_by_user = (
    train
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

# Test one user's three signals

In [16]:
sample_user = int(fair_users[0])

print(
    "Sample user:",
    sample_user
)

Sample user: 172


In [17]:
content_idx = warm_user_to_index[
    sample_user
]

content_scores = (
    user_profiles[content_idx]
    @ product_matrix.T
).toarray().ravel()

print(
    "Content score range:",
    content_scores.min(),
    content_scores.max()
)

Content score range: 0.0009299293 0.78984064


In [18]:
if sample_user in cf_user_to_idx:

    bpr_user_idx = cf_user_to_idx[
        sample_user
    ]

    user_vector = (
        bpr_user_embeddings[
            bpr_user_idx
        ]
        .numpy()
    )

    bpr_scores = (
        bpr_item_embeddings.numpy()
        @ user_vector
    )

    print(
        "BPR score range:",
        bpr_scores.min(),
        bpr_scores.max()
    )

else:

    bpr_scores = None

    print(
        "User not available in BPR"
    )

BPR score range: -0.08350612 0.08788581


In [19]:
print(
    "Popularity range:",
    popularity_scores.min(),
    popularity_scores.max()
)

Popularity range: 1 2142


# Normalization function

In [20]:
def minmax_normalize(scores):

    scores = np.asarray(
        scores,
        dtype=np.float32
    )

    min_score = np.min(scores)
    max_score = np.max(scores)

    if max_score == min_score:
        return np.ones_like(scores)

    return (
        (scores - min_score)
        / (max_score - min_score)
    )

# uild a faster BPR mapping

In [21]:
cf_idx_to_item

{0: 118401,
 1: 141657,
 2: 160984,
 3: 224700,
 4: 233200,
 5: 295653,
 6: 10034,
 7: 18519,
 8: 27248,
 9: 55710,
 10: 228229,
 11: 253720,
 12: 254301,
 13: 292240,
 14: 302810,
 15: 323030,
 16: 363106,
 17: 374400,
 18: 378862,
 19: 397068,
 20: 403998,
 21: 412628,
 22: 464731,
 23: 118551,
 24: 303828,
 25: 314073,
 26: 432478,
 27: 432992,
 28: 454422,
 29: 133814,
 30: 174894,
 31: 184990,
 32: 214189,
 33: 238447,
 34: 280029,
 35: 354233,
 36: 112445,
 37: 208932,
 38: 428725,
 39: 159380,
 40: 167127,
 41: 198996,
 42: 270291,
 43: 329624,
 44: 117889,
 45: 130121,
 46: 146364,
 47: 219718,
 48: 227688,
 49: 246589,
 50: 351798,
 51: 450404,
 52: 36504,
 53: 77930,
 54: 93001,
 55: 305031,
 56: 384758,
 57: 5951,
 58: 55239,
 59: 178480,
 60: 187848,
 61: 304558,
 62: 357779,
 63: 395273,
 64: 4537,
 65: 16370,
 66: 39717,
 67: 61442,
 68: 128218,
 69: 131882,
 70: 155550,
 71: 170500,
 72: 179290,
 73: 199647,
 74: 266439,
 75: 301563,
 76: 322503,
 77: 369998,
 78: 390881

In [22]:
cf_item_index_to_original = np.array([
    cf_idx_to_item[i]
    for i in range(len(cf_idx_to_item))
])

In [23]:
print(
    "BPR item mapping:",
    cf_item_index_to_original.shape
)

BPR item mapping: (42672,)


# Build the hybrid scorer

In [24]:
def hybrid_recommend(
    user_id,
    content_weight=0.7,
    bpr_weight=0.2,
    popularity_weight=0.1,
    candidate_k=100,
    final_k=20
):

    # 1. CANDIDATE SET

    candidates = set()

    # ------------------------------------------
    # CONTENT CANDIDATES
    # ------------------------------------------

    if user_id in warm_user_to_index:

        user_idx = warm_user_to_index[
            user_id
        ]

        content_scores_full = (
            user_profiles[user_idx]
            @ product_matrix.T
        ).toarray().ravel()

        top_content = np.argpartition(
            content_scores_full,
            -candidate_k
        )[-candidate_k:]

        candidates.update(
            product_ids[top_content]
        )

    # ------------------------------------------
    # BPR CANDIDATES
    # ------------------------------------------

    bpr_scores_full = None

    if user_id in cf_user_to_idx:

        bpr_user_idx = cf_user_to_idx[
            user_id
        ]

        user_vector = (
            bpr_user_embeddings[
                bpr_user_idx
            ]
            .numpy()
        )

        bpr_scores_full = (
            bpr_item_embeddings.numpy()
            @ user_vector
        )

        top_bpr = np.argpartition(
            bpr_scores_full,
            -candidate_k
        )[-candidate_k:]

        candidates.update(
            cf_item_index_to_original[
                top_bpr
            ]
        )

    # ------------------------------------------
    # POPULARITY CANDIDATES
    # ------------------------------------------

    candidates.update(
        popularity_scores.head(
            candidate_k
        ).index.tolist()
    )

    # ------------------------------------------
    # REMOVE SEEN ITEMS
    # ------------------------------------------

    seen = seen_items_by_user.get(
        user_id,
        set()
    )

    candidates -= seen

    candidates = list(candidates)

    if len(candidates) == 0:
        return pd.DataFrame(
            columns=[
                "item_id",
                "content_score",
                "bpr_score",
                "popularity_score",
                "hybrid_score"
            ]
        )

    # 2. GET SCORES

    candidate_indices = [
        product_id_to_index[item]
        if item in product_id_to_index
        else None
        for item in candidates
    ]

    # ------------------------------------------
    # CONTENT
    # ------------------------------------------

    content_scores = np.array([
        content_scores_full[idx]
        if idx is not None
        else 0.0
        for idx in candidate_indices
    ])

    # ------------------------------------------
    # BPR
    # ------------------------------------------

    if bpr_scores_full is not None:

        bpr_scores = np.array([
            bpr_scores_full[
                cf_item_to_idx[item]
            ]
            if item in cf_item_to_idx
            else 0.0
            for item in candidates
        ])

    else:

        bpr_scores = np.zeros(
            len(candidates)
        )

    # ------------------------------------------
    # POPULARITY
    # ------------------------------------------

    popularity_values = np.array([
        popularity_dict.get(
            item,
            0
        )
        for item in candidates
    ])

    # 3. NORMALIZE

    content_norm = minmax_normalize(
        content_scores
    )

    bpr_norm = minmax_normalize(
        bpr_scores
    )

    popularity_norm = minmax_normalize(
        popularity_values
    )

    # 4. HYBRID SCORE

    hybrid_scores = (
        content_weight
        * content_norm
        +
        bpr_weight
        * bpr_norm
        +
        popularity_weight
        * popularity_norm
    )

    # 5. TOP-K

    top_k = min(
        final_k,
        len(candidates)
    )

    top_indices = np.argsort(
        hybrid_scores
    )[::-1][:top_k]

    return pd.DataFrame({
        "item_id": np.array(
            candidates
        )[top_indices],

        "content_score": content_norm[
            top_indices
        ],

        "bpr_score": bpr_norm[
            top_indices
        ],

        "popularity_score": popularity_norm[
            top_indices
        ],

        "hybrid_score": hybrid_scores[
            top_indices
        ]
    })

# test the hybrid

In [25]:
test_recommendations = hybrid_recommend(
    sample_user,
    content_weight=0.7,
    bpr_weight=0.2,
    popularity_weight=0.1,
    candidate_k=100,
    final_k=20
)

print(test_recommendations)

    item_id  content_score  bpr_score  popularity_score  hybrid_score
0    130776       1.000000   0.632971          0.053713      0.831966
1    234198       0.972849   0.670036          0.032695      0.818271
2    328920       0.938576   0.559334          0.012611      0.770131
3    238140       0.947152   0.498460          0.000934      0.762792
4     60128       0.940568   0.498460          0.000467      0.758136
5    406296       0.922840   0.536506          0.038300      0.757119
6    455449       0.888869   0.598891          0.014946      0.743481
7    446067       0.883603   0.606380          0.025222      0.742320
8    216425       0.906579   0.508808          0.027090      0.739076
9    439258       0.861415   0.631422          0.032228      0.732498
10   201325       0.896072   0.514898          0.001868      0.730417
11   428135       0.903132   0.485817          0.004204      0.729776
12   127587       0.891028   0.498460          0.000467      0.723458
13   334395       0.

# Verify no leakage

In [26]:
seen = seen_items_by_user.get(
    sample_user,
    set()
)

overlap = (
    set(
        test_recommendations["item_id"]
    )
    &
    seen
)

print(
    "Training overlap:",
    overlap
)

Training overlap: set()


# Check the score components

In [27]:
print(
    test_recommendations[
        [
            "item_id",
            "content_score",
            "bpr_score",
            "popularity_score",
            "hybrid_score"
        ]
    ]
)

    item_id  content_score  bpr_score  popularity_score  hybrid_score
0    130776       1.000000   0.632971          0.053713      0.831966
1    234198       0.972849   0.670036          0.032695      0.818271
2    328920       0.938576   0.559334          0.012611      0.770131
3    238140       0.947152   0.498460          0.000934      0.762792
4     60128       0.940568   0.498460          0.000467      0.758136
5    406296       0.922840   0.536506          0.038300      0.757119
6    455449       0.888869   0.598891          0.014946      0.743481
7    446067       0.883603   0.606380          0.025222      0.742320
8    216425       0.906579   0.508808          0.027090      0.739076
9    439258       0.861415   0.631422          0.032228      0.732498
10   201325       0.896072   0.514898          0.001868      0.730417
11   428135       0.903132   0.485817          0.004204      0.729776
12   127587       0.891028   0.498460          0.000467      0.723458
13   334395       0.

## Define weight combinations

In [28]:
weight_configs = [
    (1.0, 0.0, 0.0),

    (0.9, 0.1, 0.0),
    (0.8, 0.2, 0.0),
    (0.7, 0.3, 0.0),

    (0.8, 0.1, 0.1),
    (0.7, 0.2, 0.1),
    (0.6, 0.3, 0.1),

    (0.7, 0.1, 0.2),
    (0.6, 0.2, 0.2),

    (0.5, 0.4, 0.1),
    (0.5, 0.3, 0.2),

    (0.4, 0.4, 0.2),
    (0.4, 0.3, 0.3),

    (0.3, 0.5, 0.2),
    (0.3, 0.4, 0.3)
]

print(
    "Configurations:",
    len(weight_configs)
)

Configurations: 15


# Generate candidates once per user

In [29]:
def get_candidate_pool(
    user_id,
    candidate_k=100
):

    candidates = set()

    # CONTENT

    content_scores_full = None

    if user_id in warm_user_to_index:

        user_idx = warm_user_to_index[
            user_id
        ]

        content_scores_full = (
            user_profiles[user_idx]
            @ product_matrix.T
        ).toarray().ravel()

        top_content = np.argpartition(
            content_scores_full,
            -candidate_k
        )[-candidate_k:]

        candidates.update(
            product_ids[top_content]
        )

    # BPR

    bpr_scores_full = None

    if user_id in cf_user_to_idx:

        bpr_user_idx = cf_user_to_idx[
            user_id
        ]

        user_vector = (
            bpr_user_embeddings[
                bpr_user_idx
            ]
            .numpy()
        )

        bpr_scores_full = (
            bpr_item_embeddings.numpy()
            @ user_vector
        )

        top_bpr = np.argpartition(
            bpr_scores_full,
            -candidate_k
        )[-candidate_k:]

        candidates.update(
            cf_item_index_to_original[
                top_bpr
            ]
        )

    # POPULARITY

    candidates.update(
        popularity_scores
        .head(candidate_k)
        .index
        .tolist()
    )

    # REMOVE SEEN

    seen = seen_items_by_user.get(
        user_id,
        set()
    )

    candidates -= seen

    return (
        list(candidates),
        content_scores_full,
        bpr_scores_full
    )

# Build cached candidate data

In [30]:
hybrid_cache = {}

for count, user_id in enumerate(
    fair_users,
    start=1
):

    candidates, content_full, bpr_full = (
        get_candidate_pool(
            user_id,
            candidate_k=100
        )
    )

    hybrid_cache[user_id] = (
        candidates,
        content_full,
        bpr_full
    )

    if count % 100 == 0:
        print(
            f"Cached {count}/{len(fair_users)} users"
        )

Cached 100/2047 users
Cached 200/2047 users
Cached 300/2047 users
Cached 400/2047 users
Cached 500/2047 users
Cached 600/2047 users
Cached 700/2047 users
Cached 800/2047 users
Cached 900/2047 users
Cached 1000/2047 users
Cached 1100/2047 users
Cached 1200/2047 users
Cached 1300/2047 users
Cached 1400/2047 users
Cached 1500/2047 users
Cached 1600/2047 users
Cached 1700/2047 users
Cached 1800/2047 users
Cached 1900/2047 users
Cached 2000/2047 users


# Evaluate a hybrid configuration

In [31]:
def evaluate_hybrid_config(
    content_weight,
    bpr_weight,
    popularity_weight,
    k=10
):

    ndcg_scores = []
    hit_scores = []
    precision_scores = []
    recall_scores = []

    for user_id in fair_users:

        candidates, content_full, bpr_full = (
            hybrid_cache[user_id]
        )

        content_scores = []
        bpr_scores = []
        popularity_values = []

        for item_id in candidates:

            # Content
            if (
                content_full is not None
                and item_id in product_id_to_index
            ):

                content_scores.append(
                    content_full[
                        product_id_to_index[item_id]
                    ]
                )

            else:
                content_scores.append(0.0)

            # BPR
            if (
                bpr_full is not None
                and item_id in cf_item_to_idx
            ):

                bpr_scores.append(
                    bpr_full[
                        cf_item_to_idx[item_id]
                    ]
                )

            else:
                bpr_scores.append(0.0)

            # Popularity
            popularity_values.append(
                popularity_dict.get(
                    item_id,
                    0
                )
            )

        content_scores = minmax_normalize(
            content_scores
        )

        bpr_scores = minmax_normalize(
            bpr_scores
        )

        popularity_values = minmax_normalize(
            popularity_values
        )

        hybrid_scores = (
            content_weight * content_scores
            +
            bpr_weight * bpr_scores
            +
            popularity_weight *
            popularity_values
        )

        top_indices = np.argsort(
            hybrid_scores
        )[::-1][:k]

        recommended = [
            candidates[i]
            for i in top_indices
        ]

        relevant = fair_ground_truth[
            user_id
        ]

        precision_scores.append(
            precision_at_k(
                recommended,
                relevant,
                k
            )
        )

        recall_scores.append(
            recall_at_k(
                recommended,
                relevant,
                k
            )
        )

        ndcg_scores.append(
            ndcg_at_k(
                recommended,
                relevant,
                k
            )
        )

        hit_scores.append(
            hit_rate_at_k(
                recommended,
                relevant,
                k
            )
        )

    return {
        "Content": content_weight,
        "BPR": bpr_weight,
        "Popularity": popularity_weight,
        "Precision@10": np.mean(
            precision_scores
        ),
        "Recall@10": np.mean(
            recall_scores
        ),
        "NDCG@10": np.mean(
            ndcg_scores
        ),
        "HitRate@10": np.mean(
            hit_scores
        )
    }

# Test ONE configuration first

In [32]:
def precision_at_k(recommended, relevant, k):
    if k == 0:
        return 0.0

    hits = sum(
        item in relevant
        for item in recommended[:k]
    )

    return hits / k


def recall_at_k(recommended, relevant, k):
    if len(relevant) == 0:
        return 0.0

    hits = sum(
        item in relevant
        for item in recommended[:k]
    )

    return hits / len(relevant)


def hit_rate_at_k(recommended, relevant, k):
    return float(
        any(
            item in relevant
            for item in recommended[:k]
        )
    )


def ndcg_at_k(recommended, relevant, k):

    if len(relevant) == 0:
        return 0.0

    dcg = 0.0

    for rank, item in enumerate(
        recommended[:k],
        start=1
    ):
        if item in relevant:
            dcg += 1 / np.log2(rank + 1)

    ideal_hits = min(
        len(relevant),
        k
    )

    idcg = sum(
        1 / np.log2(rank + 1)
        for rank in range(
            1,
            ideal_hits + 1
        )
    )

    if idcg == 0:
        return 0.0

    return dcg / idcg

In [33]:
test_result = evaluate_hybrid_config(
    0.7,
    0.2,
    0.1,
    k=10
)

print(test_result)

{'Content': 0.7, 'BPR': 0.2, 'Popularity': 0.1, 'Precision@10': 0.008353688324377138, 'Recall@10': 0.022541401780989505, 'NDCG@10': 0.01705437712611094, 'HitRate@10': 0.0635075720566683}


## Run the weight search

In [34]:
weight_results = []

for content_w, bpr_w, popularity_w in weight_configs:

    print(
        f"\nTesting: "
        f"Content={content_w}, "
        f"BPR={bpr_w}, "
        f"Popularity={popularity_w}"
    )

    result = evaluate_hybrid_config(
        content_w,
        bpr_w,
        popularity_w,
        k=10
    )

    weight_results.append(result)

weight_results_df = pd.DataFrame(
    weight_results
)

weight_results_df = (
    weight_results_df
    .sort_values(
        "NDCG@10",
        ascending=False
    )
    .reset_index(drop=True)
)

print(weight_results_df)


Testing: Content=1.0, BPR=0.0, Popularity=0.0

Testing: Content=0.9, BPR=0.1, Popularity=0.0

Testing: Content=0.8, BPR=0.2, Popularity=0.0

Testing: Content=0.7, BPR=0.3, Popularity=0.0

Testing: Content=0.8, BPR=0.1, Popularity=0.1

Testing: Content=0.7, BPR=0.2, Popularity=0.1

Testing: Content=0.6, BPR=0.3, Popularity=0.1

Testing: Content=0.7, BPR=0.1, Popularity=0.2

Testing: Content=0.6, BPR=0.2, Popularity=0.2

Testing: Content=0.5, BPR=0.4, Popularity=0.1

Testing: Content=0.5, BPR=0.3, Popularity=0.2

Testing: Content=0.4, BPR=0.4, Popularity=0.2

Testing: Content=0.4, BPR=0.3, Popularity=0.3

Testing: Content=0.3, BPR=0.5, Popularity=0.2

Testing: Content=0.3, BPR=0.4, Popularity=0.3
    Content  BPR  Popularity  Precision@10  Recall@10   NDCG@10  HitRate@10
0       0.4  0.4         0.2      0.009233   0.024343  0.019391    0.070347
1       0.4  0.3         0.3      0.009233   0.023664  0.018756    0.071812
2       0.5  0.3         0.2      0.008891   0.022905  0.018583    

# Find the best configuration

In [35]:
best_hybrid = (
    weight_results_df.iloc[0]
)

print(
    "BEST HYBRID CONFIGURATION"
)

print(
    "Content:",
    best_hybrid["Content"]
)

print(
    "BPR:",
    best_hybrid["BPR"]
)

print(
    "Popularity:",
    best_hybrid["Popularity"]
)

print(
    "NDCG@10:",
    best_hybrid["NDCG@10"]
)

BEST HYBRID CONFIGURATION
Content: 0.4
BPR: 0.4
Popularity: 0.2
NDCG@10: 0.019391284553690696


In [36]:
test = pd.read_parquet(
    "../data/processed/test.parquet"
)

print("Test:", test.shape)
print(
    "Test users:",
    test["user_id"].nunique()
)

Test: (413347, 5)
Test users: 233728


In [37]:
test_ground_truth = (
    test[
        test["user_id"].isin(fair_users)
    ]
    .groupby("user_id")["item_id"]
    .apply(set)
    .to_dict()
)

test_eval_users = np.array(
    sorted(test_ground_truth.keys())
)

print(
    "Test evaluation users:",
    len(test_eval_users)
)

print(
    "Test interactions:",
    sum(
        len(v)
        for v in test_ground_truth.values()
    )
)

Test evaluation users: 705
Test interactions: 8826


In [38]:
print(
    "Users with test ground truth:",
    len(test_eval_users)
)

print(
    "Users without test interactions:",
    len(
        set(fair_users)
        - set(test_eval_users)
    )
)

Users with test ground truth: 705
Users without test interactions: 1342


## Create a test-specific cache

In [39]:
test_hybrid_cache = {}

for count, user_id in enumerate(
    test_eval_users,
    start=1
):

    candidates, content_full, bpr_full = (
        get_candidate_pool(
            user_id,
            candidate_k=100
        )
    )

    test_hybrid_cache[user_id] = (
        candidates,
        content_full,
        bpr_full
    )

    if count % 100 == 0:
        print(
            f"Cached {count}/{len(test_eval_users)} test users"
        )

print(
    f"Finished caching {len(test_eval_users)} test users"
)

Cached 100/705 test users
Cached 200/705 test users
Cached 300/705 test users
Cached 400/705 test users
Cached 500/705 test users
Cached 600/705 test users
Cached 700/705 test users
Finished caching 705 test users


# Evaluate the LOCKED hybrid

In [40]:
def evaluate_test_hybrid(
    content_weight=0.4,
    bpr_weight=0.4,
    popularity_weight=0.2,
    ks=(5, 10, 20)
):

    results = {
        k: {
            "precision": [],
            "recall": [],
            "ndcg": [],
            "hit": []
        }
        for k in ks
    }

    for count, user_id in enumerate(
        test_eval_users,
        start=1
    ):

        candidates, content_full, bpr_full = (
            test_hybrid_cache[user_id]
        )

        content_scores = []
        bpr_scores = []
        popularity_values = []

        for item_id in candidates:

            # Content score
            if (
                content_full is not None
                and item_id in product_id_to_index
            ):
                content_scores.append(
                    content_full[
                        product_id_to_index[item_id]
                    ]
                )
            else:
                content_scores.append(0.0)

            # BPR score
            if (
                bpr_full is not None
                and item_id in cf_item_to_idx
            ):
                bpr_scores.append(
                    bpr_full[
                        cf_item_to_idx[item_id]
                    ]
                )
            else:
                bpr_scores.append(0.0)

            # Popularity
            popularity_values.append(
                popularity_dict.get(
                    item_id,
                    0
                )
            )

        # Normalize each signal
        content_norm = minmax_normalize(
            content_scores
        )

        bpr_norm = minmax_normalize(
            bpr_scores
        )

        popularity_norm = minmax_normalize(
            popularity_values
        )

        # Locked hybrid weights
        hybrid_scores = (
            0.4 * content_norm
            +
            0.4 * bpr_norm
            +
            0.2 * popularity_norm
        )

        for k in ks:

            top_indices = np.argsort(
                hybrid_scores
            )[::-1][:k]

            recommended = [
                candidates[i]
                for i in top_indices
            ]

            relevant = test_ground_truth[
                user_id
            ]

            results[k]["precision"].append(
                precision_at_k(
                    recommended,
                    relevant,
                    k
                )
            )

            results[k]["recall"].append(
                recall_at_k(
                    recommended,
                    relevant,
                    k
                )
            )

            results[k]["ndcg"].append(
                ndcg_at_k(
                    recommended,
                    relevant,
                    k
                )
            )

            results[k]["hit"].append(
                hit_rate_at_k(
                    recommended,
                    relevant,
                    k
                )
            )

        if count % 100 == 0:
            print(
                f"Evaluated "
                f"{count}/{len(test_eval_users)} users"
            )

    return pd.DataFrame([
        {
            "Model": "Hybrid",
            "K": k,
            "Precision@K":
                np.mean(
                    results[k]["precision"]
                ),
            "Recall@K":
                np.mean(
                    results[k]["recall"]
                ),
            "NDCG@K":
                np.mean(
                    results[k]["ndcg"]
                ),
            "HitRate@K":
                np.mean(
                    results[k]["hit"]
                )
        }
        for k in ks
    ])

In [41]:
final_hybrid_test = evaluate_test_hybrid(
    content_weight=0.4,
    bpr_weight=0.4,
    popularity_weight=0.2,
    ks=(5, 10, 20)
)

print(final_hybrid_test)

Evaluated 100/705 users
Evaluated 200/705 users
Evaluated 300/705 users
Evaluated 400/705 users
Evaluated 500/705 users
Evaluated 600/705 users
Evaluated 700/705 users
    Model   K  Precision@K  Recall@K    NDCG@K  HitRate@K
0  Hybrid   5     0.008794  0.011829  0.013930   0.038298
1  Hybrid  10     0.007518  0.016739  0.015367   0.055319
2  Hybrid  20     0.005532  0.024252  0.017249   0.072340


## Evaluate Content-Based on the same 705 test users

In [42]:
def evaluate_content_test(
    users,
    ground_truth,
    k=10
):

    precision_scores = []
    recall_scores = []
    ndcg_scores = []
    hit_scores = []

    for count, user_id in enumerate(users, start=1):

        if user_id not in warm_user_to_index:
            continue

        user_idx = warm_user_to_index[user_id]

        scores = (
            user_profiles[user_idx]
            @ product_matrix.T
        ).toarray().ravel()

        # Remove training items
        seen = seen_items_by_user.get(
            user_id,
            set()
        )

        for item_id in seen:

            idx = product_id_to_index.get(item_id)

            if idx is not None:
                scores[idx] = -np.inf

        top_indices = np.argpartition(
            scores,
            -k
        )[-k:]

        top_indices = top_indices[
            np.argsort(
                scores[top_indices]
            )[::-1]
        ]

        recommended = [
            product_ids[i]
            for i in top_indices
        ]

        relevant = ground_truth[user_id]

        precision_scores.append(
            precision_at_k(
                recommended,
                relevant,
                k
            )
        )

        recall_scores.append(
            recall_at_k(
                recommended,
                relevant,
                k
            )
        )

        ndcg_scores.append(
            ndcg_at_k(
                recommended,
                relevant,
                k
            )
        )

        hit_scores.append(
            hit_rate_at_k(
                recommended,
                relevant,
                k
            )
        )

        if count % 100 == 0:
            print(
                f"Evaluated {count}/{len(users)}"
            )

    return {
        "Precision@10": np.mean(precision_scores),
        "Recall@10": np.mean(recall_scores),
        "NDCG@10": np.mean(ndcg_scores),
        "HitRate@10": np.mean(hit_scores)
    }

# Evaluate Popularity on the same users

In [43]:
def evaluate_popularity_test(
    users,
    ground_truth,
    k=10
):

    precision_scores = []
    recall_scores = []
    ndcg_scores = []
    hit_scores = []

    popularity_ranking = (
        popularity_scores.index
        .tolist()
    )

    for count, user_id in enumerate(
        users,
        start=1
    ):

        seen = seen_items_by_user.get(
            user_id,
            set()
        )

        recommended = [
            item
            for item in popularity_ranking
            if item not in seen
        ][:k]

        relevant = ground_truth[user_id]

        precision_scores.append(
            precision_at_k(
                recommended,
                relevant,
                k
            )
        )

        recall_scores.append(
            recall_at_k(
                recommended,
                relevant,
                k
            )
        )

        ndcg_scores.append(
            ndcg_at_k(
                recommended,
                relevant,
                k
            )
        )

        hit_scores.append(
            hit_rate_at_k(
                recommended,
                relevant,
                k
            )
        )

    return {
        "Precision@10": np.mean(precision_scores),
        "Recall@10": np.mean(recall_scores),
        "NDCG@10": np.mean(ndcg_scores),
        "HitRate@10": np.mean(hit_scores)
    }

In [44]:
popularity_test_result = evaluate_popularity_test(
    test_eval_users,
    test_ground_truth,
    k=10
)

print(popularity_test_result)

{'Precision@10': 0.0015602836879432625, 'Recall@10': 0.00016683035113913074, 'NDCG@10': 0.0022726089088690575, 'HitRate@10': 0.01276595744680851}


# BPR test evaluation

In [45]:
def evaluate_bpr_test(
    users,
    ground_truth,
    k=10
):

    precision_scores = []
    recall_scores = []
    ndcg_scores = []
    hit_scores = []

    item_matrix = (
        bpr_item_embeddings
        .numpy()
    )

    for count, user_id in enumerate(
        users,
        start=1
    ):

        if user_id not in cf_user_to_idx:
            continue

        user_idx = cf_user_to_idx[
            user_id
        ]

        user_vector = (
            bpr_user_embeddings[
                user_idx
            ]
            .numpy()
        )

        scores = (
            item_matrix
            @ user_vector
        )

        # BPR items only
        top_indices = np.argpartition(
            scores,
            -k
        )[-k:]

        top_indices = top_indices[
            np.argsort(
                scores[top_indices]
            )[::-1]
        ]

        recommended = [
            cf_item_index_to_original[i]
            for i in top_indices
        ]

        # Remove training items
        seen = seen_items_by_user.get(
            user_id,
            set()
        )

        recommended = [
            item
            for item in recommended
            if item not in seen
        ][:k]

        relevant = ground_truth[user_id]

        precision_scores.append(
            precision_at_k(
                recommended,
                relevant,
                k
            )
        )

        recall_scores.append(
            recall_at_k(
                recommended,
                relevant,
                k
            )
        )

        ndcg_scores.append(
            ndcg_at_k(
                recommended,
                relevant,
                k
            )
        )

        hit_scores.append(
            hit_rate_at_k(
                recommended,
                relevant,
                k
            )
        )

        if count % 100 == 0:
            print(
                f"Evaluated {count}/{len(users)}"
            )

    return {
        "Precision@10": np.mean(precision_scores),
        "Recall@10": np.mean(recall_scores),
        "NDCG@10": np.mean(ndcg_scores),
        "HitRate@10": np.mean(hit_scores)
    }

In [46]:
bpr_test_result = evaluate_bpr_test(
    test_eval_users,
    test_ground_truth,
    k=10
)

print(bpr_test_result)

Evaluated 100/705
Evaluated 200/705
Evaluated 300/705
Evaluated 400/705
Evaluated 500/705
Evaluated 600/705
Evaluated 700/705
{'Precision@10': 0.002411347517730497, 'Recall@10': 0.003171012265364757, 'NDCG@10': 0.00445540869378512, 'HitRate@10': 0.018439716312056736}


# Create the final test leaderboard

In [47]:
print("evaluate_content_test" in globals())

True


In [48]:
content_test_result = evaluate_content_test(
    test_eval_users,
    test_ground_truth,
    k=10
)

print(content_test_result)

Evaluated 100/705
Evaluated 200/705
Evaluated 300/705
Evaluated 400/705
Evaluated 500/705
Evaluated 600/705
Evaluated 700/705
{'Precision@10': 0.005531914893617022, 'Recall@10': 0.01685877960429069, 'NDCG@10': 0.011305017539602202, 'HitRate@10': 0.04397163120567376}


In [49]:
print("popularity_test_result" in globals())
print("bpr_test_result" in globals())
print("content_test_result" in globals())

True
True
True


In [50]:
final_test_comparison = pd.DataFrame([
    {
        "Model": "Popularity",
        **popularity_test_result
    },
    {
        "Model": "Content-Based",
        **content_test_result
    },
    {
        "Model": "BPR",
        **bpr_test_result
    },
    {
        "Model": "Hybrid",
        "Precision@10": 0.007518,
        "Recall@10": 0.016739,
        "NDCG@10": 0.015367,
        "HitRate@10": 0.055319
    }
])

print(
    final_test_comparison.sort_values(
        "NDCG@10",
        ascending=False
    )
)

           Model  Precision@10  Recall@10   NDCG@10  HitRate@10
3         Hybrid      0.007518   0.016739  0.015367    0.055319
1  Content-Based      0.005532   0.016859  0.011305    0.043972
2            BPR      0.002411   0.003171  0.004455    0.018440
0     Popularity      0.001560   0.000167  0.002273    0.012766
